# Thread workers

> Threaded workers and their queues to isolate user-interfaces from IO operations and CPU intensive tasks into a background thread

In [ ]:
#| default_exp threadworkers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
#| hide

import threading
import queue
import time
import logging

from typing import NamedTuple, Callable, Any

from edcompanion.core import configuration


In [ ]:
#| hide
from edcompanion.core import init_console_logging
import pandas as pd

In [ ]:
#| hide

init_console_logging(__name__)

2025-12-13T10:37:02+0100 INFO	79884	__main__	core.py	init_console_logging	41	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
#| hide

syslog = logging.getLogger(__name__)


In [ ]:

#| hide
syslog.info("Hello World")

2025-12-13T10:37:02+0100 INFO	79884	__main__	1748110922.py	<module>	2	Hello World


#### Queues, blocking and timeouts

We want the user-inteface to run free, without ever waiting for potentially long running tasks. Queues isolate the user-interface from the background tasks. 

FiFo queues get (pop) and put (push) items in the order they are added. At each of these interactions the caller can wait for the queue or ignore it's state depending, typically on instructions to block or timeout:


In [ ]:
#| echo: false
#| output: true
pd.DataFrame({
    'block':   [True, False, True, False],
    'timeout': ['None', 'None', '> 0', '> 0'],
    'on empty':['Wait', 'raise Empty', 'Wait timeout then raise Empty ', 'raise Empty'],
    'on full': ['Block', 'raise Full', 'Block timeout then raise Full', 'raise Full'],}
).set_index(['block', 'timeout']).sort_index(ascending=False)


on empty                        on full
block timeout                                                               
True  None                               Wait                          Block
      > 0      Wait timeout then raise Empty   Block timeout then raise Full
False None                        raise Empty                     raise Full
      > 0                         raise Empty                     raise Full

In [ ]:
#| export

class QueueGetArgs(NamedTuple):
    block: bool
    timeout: float

def _create_queue_get_args(block, timeout):
    return QueueGetArgs(block=block, timeout=timeout)

class QueuePutArgs(NamedTuple):
    block: bool
    timeout: float

def _create_queue_put_args(block, timeout):
    return QueuePutArgs(block=block, timeout=timeout)

#### `WorkerArgs`

A NamedTuple class as item to hold args and kwargs for a task execution

In [ ]:
#| export
class WorkerArgs(NamedTuple):
    args: tuple
    kwargs: dict


#### `_bind_queue_put`

Binds a queue with standard kwargs for queue.put() work items into the queue.



In [ ]:
#| export

def _bind_queue_put_nowait(
        task_queue,     # The queue to put work items into
) -> Callable[[WorkerArgs], None]:
    
    "Binds a queue with kwargs for putting work items non-blocking into the queue"

    def put_task_item(*work_args, **work_kwargs):
        task_queue.put_nowait(item=WorkerArgs(work_args, work_kwargs))

    return put_task_item



In [ ]:
#| export

def _bind_queue_put(
        task_queue,     # The queue to put work items into
        block = True,   # Whether to block on full queue
        timeout = None, # Timeout for block
) -> Callable[[WorkerArgs], None]:
    
    "Binds a queue with kwargs for putting work items into the queue -defaults to blocking"

    def put_task_item(*work_args, **work_kwargs):
        task_queue.put(item=WorkerArgs(work_args, work_kwargs), block=block, timeout=timeout)

    return put_task_item


#### `_bind_queue_get`

In [ ]:
#| export
def _bind_queue_get(
        product_queue,      # The queue to get produced items from
        block = True,   # Whether to block on full queue
        timeout = None, # Timeout for block
) -> Callable[[], WorkerArgs]:
    
    "Binds a queue and kwargs for getting produced items from the queue"

    def get_task_item():
        try:
            item = product_queue.get(block=block, timeout=timeout)
            product_queue.task_done()

        except queue.Empty:
            item = None

        return item

    return get_task_item

In [ ]:
#| export
def _bind_queue_get_nowait(
        product_queue,      # The queue to get produced items from
) -> Callable[[], WorkerArgs]:
    
    "Binds a queue and kwargs for getting produced items from the queue"

    def get_task_item():
        try:
            item = product_queue.get_nowait()
            product_queue.task_done()

        except queue.Empty:
            item = None

        return item

    return get_task_item

In [ ]:
test_queue = queue.Queue(maxsize=100)

put_item = _bind_queue_put(test_queue, block=False)
get_item = _bind_queue_get(test_queue, block=False)


In [ ]:

for i in range(5):
    put_item(i, str(i*i), sep='\t')


In [ ]:
while True:
    item = get_item()
    if item is None:
        break
    print(*item.args, **item.kwargs)

0	0
1	1
2	4
3	9
4	16


In [ ]:
#| export

def bind_queue_as_generator_blocking(
        source_queue       # The queue to get produced items from
) -> callable:
    
    "Binds a queue for getting produced items from the queue"

    def get_task_items():
        try:
            while True:
                item = source_queue.get(block=True, timeout=None)
                source_queue.task_done()
                yield item

        except queue.Empty:
            pass

    return get_task_items

In [ ]:
#| export

def bind_queue_as_generator_nowait(
        source_queue       # The queue to get produced items from
) -> callable:
    
    "Binds a queue for getting produced items from the queue"

    def get_task_items():
        try:
            while True:
                item = source_queue.get_nowait()
                source_queue.task_done()
                yield item

        except queue.Empty:
            pass

    return get_task_items

In [ ]:
test_queue = queue.Queue(maxsize=10)

get_items = bind_task_queue_generator(test_queue)


In [ ]:
for i in range(5):
    test_queue.put(i)


In [ ]:
list(get_items())



[0, 1, 2, 3, 4]

In [ ]:
list(get_items())


[]

In [ ]:
def bind_bidirectional_queues(
        worker_queue,      # The queue to put work items into
        product_queue,      # The queue to get produced items from
        
) -> callable:
    
    def generator():
        workargs = None
        
        while True:
            try:
                while workargs is None:
                    workargs = yield product_queue.get_nowait()
            
            except queue.Empty:
                workargs = yield None

            if workargs is not None:
                worker_queue.put(workargs)
                

In [ ]:
#| exporti

def _create_sleeper(base=0.1):

    class Sleeper(NamedTuple):
        sleep: Callable
        reset: Callable

    sleepfor = base
    sleepmax = 5*sleepfor

    def _sleep():
        time.sleep(sleepfor)
        sleepfor = min(sleepfor * 1.2, sleepmax)
        
    def _reset():
        sleepfor = 0.05

    return Sleeper(_sleep, _reset)


In [ ]:
sleeper = _create_sleeper()
sleeper.sleep()

UnboundLocalError: cannot access local variable 'sleepfor' where it is not associated with a value

In [ ]:
#| exporti

def workloop(
        task_queue: queue.Queue,
        product_queue: queue.Queue,
        worker_fn:  callable,
        stop_event: threading.Event,
    ):
        """
        A worker thread loop that gets items from a queue, processes 
        them with a worker function and puts it's result on a queue.
        """

        sleepfor = 0.05 # initial sleep time when no work is available
        sleepmax = 0.5  # max sleep time when no work is available

        def _sleep():
            time.sleep(sleepfor)
            sleepfor = min(sleepfor * 1.2, sleepmax)

        # indefinately keep getting new items from the queue to process
        while True:
            
            try:
                item = None
                try:
                    # by default queue.get() waits for new items
                    item = task_queue.get_nowait()
                    task_queue.task_done()
                    sleepfor = 0.05

                except queue.Empty:
                    if stop_event.is_set():
                        break
                    else:
                        _sleep()

                result = worker_fn(item)

                while True:
                    try:
                        product_queue.put_nowait(result)

                    except queue.Full:
                        if stop_event.is_set():
                            break
                        else:
                            _sleep()


            except Exception as x:
                syslog.exception("Exception: %s", x, exc_info=True, stack_info=True)



In [ ]:
import nbdev; nbdev.nbdev_export()